[![pythonista.io](img/pythonista.png)](https://www.pythonista.io)

# DevSecOps, workflows de repositorio y entornos.

Este cuaderno funciona como capitulo marco para Py271. Su objetivo es fijar el lenguaje comun del curso antes de entrar a pipelines, quality gates, despliegues, Terraform y operacion segura.

## Objetivos.

- Entender que significa DevSecOps y por que no se reduce a agregar escaneos al pipeline.
- Identificar los workflows de repositorio mas comunes y sus efectos sobre CI/CD.
- Distinguir el papel de los entornos `dev`, `test` y `prod` dentro de una cadena de entrega segura.

## Que es DevSecOps.

DevSecOps es una forma de organizar el desarrollo y la operacion de software en la que seguridad, calidad, entrega e infraestructura forman parte del mismo sistema de trabajo. No significa mover la seguridad a un equipo aislado al final del proceso, sino integrarla desde el diseno, el desarrollo, la validacion, el build, el despliegue y la observabilidad.

En la practica, DevSecOps implica:

- automatizar verificaciones de calidad y seguridad lo mas pronto posible;
- tratar la infraestructura y las politicas como codigo versionado;
- reducir credenciales estaticas y privilegios excesivos;
- dejar evidencia verificable de como se construyo y desplego un artefacto;
- establecer retroalimentacion rapida cuando algo falla.

La idea central es que seguridad no sea una puerta al final del proceso, sino una propiedad verificable de toda la cadena de entrega.

## DevOps y DevSecOps.

DevOps pone el foco en colaboracion, automatizacion y entrega continua entre desarrollo y operacion. DevSecOps conserva esos principios, pero agrega una exigencia: toda decision tecnica relevante debe considerar impacto de seguridad y trazabilidad.

En un flujo DevSecOps maduro, preguntas como estas no se dejan para el cierre:

- quien puede promover cambios a produccion;
- con que identidad autentica el pipeline frente a la nube;
- si una imagen o paquete tiene SBOM y provenance;
- si los secretos viven en el lugar correcto;
- si la infraestructura fue creada manualmente o por codigo auditado.

## Esquema visual: DevSecOps como cadena de control.

```text
Cambio en repo
   |
   v
Calidad tecnica -> Seguridad -> Build -> Evidencia -> Despliegue -> Observabilidad
   |                |            |         |             |               |
   |                |            |         |             |               |
 lint/tests      escaneos      imagen     SBOM      entornos        smoke tests
 tipado          secretos      o paquete  provenance  aprobaciones   rollback
```

La idea del esquema es simple: cada fase agrega una capa de confianza. Si una fase falla, la promocion debe detenerse.

## Workflows de repositorio.

Un workflow de repositorio describe como fluye el cambio desde que alguien modifica codigo hasta que ese cambio puede integrarse, liberarse o desplegarse. No debe confundirse con un workflow de GitHub Actions: el primero es una politica de colaboracion del repositorio; el segundo es una automatizacion concreta dentro de esa politica.

Algunos patrones comunes son:

- `feature branches`: cada cambio vive en una rama corta y se integra por pull request;
- `trunk-based development`: ramas muy cortas y merges frecuentes hacia una rama principal protegida;
- `release branches`: se separan ramas para estabilizar una version antes de liberarla;
- `hotfix branches`: permiten corregir incidentes criticos sin mezclar todos los cambios en curso.

Py271 se beneficia de una estrategia simple: ramas cortas, pull requests, proteccion de rama principal y checks obligatorios antes del merge. Esa combinacion favorece trazabilidad y reduce riesgo operacional.

## Politicas de repositorio que importan.

Para que un workflow de repositorio soporte CI/CD seguro, conviene definir algunas reglas explicitas:

- la rama principal no debe aceptar cambios directos sin revision;
- los pull requests deben disparar checks minimos de calidad;
- los workflows sensibles deben vivir bajo proteccion y revision;
- las liberaciones deben quedar asociadas a tags o versiones trazables;
- los cambios de infraestructura y despliegue no deben tener el mismo nivel de libertad que un cambio de texto o documentacion.

Estas reglas son operativas, no burocraticas: buscan que el repositorio refleje el nivel de control que se espera del sistema que luego sera desplegado.

## Entornos: dev, test y prod.

Los entornos separan fases del ciclo de entrega. No son solo nombres convenientes; representan niveles distintos de riesgo, estabilidad, datos, permisos y controles.

### `dev`

Es el entorno de iteracion rapida. Se usa para validar cambios tempranos, explorar configuraciones y comprobar que una version funciona en condiciones cercanas a la realidad, pero con menor costo y menor restriccion.

### `test`

Es el entorno donde se verifican comportamiento, integracion y criterios de aceptacion con mayor disciplina. Puede incluir datos sinteticos, servicios simulados, pruebas automatizadas y validaciones mas cercanas a un despliegue real.

### `prod`

Es el entorno que recibe trafico o uso real. Por eso exige mayor estabilidad, aprobaciones, observabilidad, control de secretos, identidades acotadas y mecanismos claros de rollback.

## Esquema visual: promocion entre entornos.

```text
feature branch
      |
      v
pull request
      |
      v
checks en CI
      |
      v
merge a main
      |
      v
deploy a dev -> validacion -> deploy a test -> aprobacion -> deploy a prod
```

Este recorrido muestra que el mismo cambio no deberia saltar directo a produccion sin pasar por controles progresivos.

## Promocion entre entornos.

La promocion sana no consiste en reconstruir artefactos distintos en cada entorno, sino en promover el mismo artefacto validado a traves de controles diferentes. La construccion debe ser reproducible; lo que cambia entre `dev`, `test` y `prod` son variables, permisos, aprobaciones, datos y restricciones operativas.

Ese principio evita una falla comun: probar una cosa y desplegar otra.

## Tabla rapida de entornos.

| Entorno | Objetivo principal | Riesgo aceptable | Controles tipicos |
| --- | --- | --- | --- |
| `dev` | iteracion y validacion temprana | medio | variables basicas, despliegue rapido, datos no productivos |
| `test` | validacion funcional e integracion | medio-bajo | pruebas automatizadas, datos sinteticos, checks adicionales |
| `prod` | operacion real | bajo | aprobaciones, observabilidad, secretos controlados, rollback |

## Relacion entre DevSecOps, repositorio y entornos.

Estos tres conceptos se refuerzan entre si:

- DevSecOps define la necesidad de integrar seguridad y trazabilidad en toda la cadena;
- el workflow del repositorio define como se acepta y promueve el cambio;
- los entornos definen bajo que condiciones operativas puede ejecutarse ese cambio.

Sin workflow de repositorio claro, el pipeline se vuelve inconsistente. Sin entornos claros, el despliegue pierde control. Sin enfoque DevSecOps, la automatizacion puede ser rapida pero insegura.

In [ ]:
flujo = {
    'repositorio': ['branch', 'pull_request', 'merge'],
    'pipeline': ['lint', 'tests', 'build', 'scan'],
    'entornos': ['dev', 'test', 'prod'],
}

for capa, pasos in flujo.items():
    print(capa, '=>', ' -> '.join(pasos))

## Cierre.

Py271 parte de esta idea: un pipeline seguro no es solo una secuencia de comandos. Es la expresion automatizada de una politica tecnica sobre cambios, identidades, infraestructura, artefactos y despliegues.

Los siguientes cuadernos aterrizan cada parte de ese sistema.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>